# 확률분포 시 생성기 LoRA 파인튜닝 Colab 노트북

- base model: `Qwen/Qwen2.5-0.5B-Instruct`
- Colab T4 기준 QLoRA
- 입력: 사용자 경험 프롬프트
- 출력: 정확히 3줄 한국어 시
- 데이터셋: `data/train.jsonl`의 `messages` 형식
- 현재 기준 데이터셋: classic 200개 + modern 50개 = 250개
- adapter와 merged model을 모두 Hugging Face Hub에 업로드


## 1. 라이브러리 설치
최신 TRL의 `SFTConfig` 방식으로 학습한다.

In [ ]:
!pip -q install -U "transformers>=4.56.0" "accelerate>=1.10.0" "peft>=0.17.0" "datasets>=4.0.0" "trl>=0.24.0" bitsandbytes huggingface_hub sentencepiece


## 2. Hugging Face 로그인

In [ ]:
from huggingface_hub import login
login()


## 3. 기본 설정

In [ ]:
import os
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

BASE_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
ADAPTER_REPO_ID = "z-unghyun/poem-generator-lora-adapter"
MERGED_REPO_ID = "z-unghyun/poem-generator-merged"
OUTPUT_DIR = "./outputs/poem-generator-lora"
MERGED_OUTPUT_DIR = "./outputs/poem-generator-merged"

# 기본값은 GitHub main의 data/train.jsonl을 직접 불러오는 방식이다.
# 로컬 파일을 직접 업로드하려면 DATA_MODE = "local_upload"로 바꾼다.
DATA_MODE = "github_raw"  # "github_raw", "local_upload", or "hub_dataset"
GITHUB_RAW_TRAIN_URL = "https://raw.githubusercontent.com/z-unghyun/poem-generator/main/data/train.jsonl"
LOCAL_TRAIN_PATH = "train.jsonl"
HF_DATASET_ID = "z-unghyun/poem-generator-dataset"
HF_DATA_FILE = "train.jsonl"
EXPECTED_ROWS = 250

LORA_TARGET_MODULES = ["q_proj", "v_proj"]
# 성능이 부족하면 아래처럼 확장해서 재학습한다.
# LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]
# LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

print("cuda_available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    BF16_AVAILABLE = torch.cuda.is_bf16_supported()
    print("bf16_available:", BF16_AVAILABLE)
else:
    raise RuntimeError("GPU가 잡히지 않았습니다. 런타임 유형을 T4 GPU로 바꿔주세요.")


## 4. 데이터셋 불러오기

In [ ]:
from datasets import load_dataset
from pathlib import Path

if DATA_MODE == "github_raw":
    dataset = load_dataset(
        "json",
        data_files=GITHUB_RAW_TRAIN_URL,
        split="train",
        download_mode="force_redownload",
    )
elif DATA_MODE == "local_upload":
    from google.colab import files
    if not Path(LOCAL_TRAIN_PATH).exists():
        uploaded = files.upload()
        if LOCAL_TRAIN_PATH not in uploaded:
            first_name = next(iter(uploaded.keys()))
            Path(LOCAL_TRAIN_PATH).write_bytes(uploaded[first_name])
            print(f"saved uploaded file as {LOCAL_TRAIN_PATH}")
    dataset = load_dataset("json", data_files=LOCAL_TRAIN_PATH, split="train")
elif DATA_MODE == "hub_dataset":
    dataset = load_dataset(HF_DATASET_ID, data_files=HF_DATA_FILE, split="train")
else:
    raise ValueError(f"Unknown DATA_MODE: {DATA_MODE}")

print(dataset)
print(dataset[0])


## 5. 데이터셋 검수

In [ ]:
def count_nonempty_lines(text: str) -> int:
    return len([line for line in str(text).splitlines() if line.strip()])

def validate_messages(example):
    messages = example.get("messages")
    if not isinstance(messages, list) or len(messages) != 3:
        return {"is_valid": False, "validation_reason": "messages_not_3_turns"}
    roles = [m.get("role") for m in messages]
    if roles != ["system", "user", "assistant"]:
        return {"is_valid": False, "validation_reason": f"invalid_roles:{roles}"}
    poem = messages[2].get("content", "")
    line_count = count_nonempty_lines(poem)
    if line_count != 3:
        return {"is_valid": False, "validation_reason": f"assistant_not_three_lines:{line_count}"}
    return {"is_valid": True, "validation_reason": "ok"}

checked = dataset.map(validate_messages)
bad = checked.filter(lambda x: not x["is_valid"])
print("total:", len(checked))
print("bad:", len(bad))

if len(checked) != EXPECTED_ROWS:
    raise ValueError(f"Expected exactly {EXPECTED_ROWS} rows, got {len(checked)}. data/train.jsonl이 최신 250개 버전인지 확인하세요.")
if len(bad) > 0:
    print(bad[:5])
    raise ValueError("Invalid training rows found. Fix train.jsonl before training.")

dataset = checked.remove_columns(["is_valid", "validation_reason"])


## 6. Tokenizer와 chat template 적용
`messages`를 Qwen chat template 문자열로 변환한 뒤, TRL에는 `text` 필드를 학습 대상으로 넘긴다.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def apply_chat_template(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = dataset.map(apply_chat_template)
print(dataset[0]["text"][:800])


## 7. QLoRA 모델 로드

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if BF16_AVAILABLE else torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)


## 8. LoRA 설정 및 학습
최신 TRL 방식: `SFTConfig`에 `dataset_text_field`, `max_length`, `eos_token`을 넣고 `SFTTrainer`에는 `processing_class=tokenizer`를 넘긴다.

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=LORA_TARGET_MODULES,
)

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=8,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="epoch",
    bf16=BF16_AVAILABLE,
    fp16=not BF16_AVAILABLE,
    optim="paged_adamw_8bit",
    report_to="none",
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    dataset_text_field="text",
    max_length=512,
    eos_token="<|im_end|>",
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

model = trainer.model
model.eval()


## 9. Adapter 테스트

In [ ]:
def make_prompt(experience: str):
    messages = [
        {"role": "system", "content": "너는 경험을 3줄 한국어 시로 바꾸는 시 생성 모델이다."},
        {"role": "user", "content": f"경험: {experience}"},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def clean_poem(text: str) -> str:
    text = text.strip()
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return "\n".join(lines[:3])

def generate_poem(experience: str, max_new_tokens: int = 80):
    prompt = make_prompt(experience)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.08,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    return clean_poem(generated)

test_inputs = [
    "비 오는 밤 버스 정류장에서 혼자 기다림",
    "새벽에 과제를 하다가 모니터 빛에 눈이 아픔",
    "보고 싶은 사람이 올 때를 기다리며 긴 밤을 견딤",
    "수영을 마치고 젖은 머리로 집에 돌아옴",
    "AI 프로젝트를 하다가 언어가 이상하게 무너지는 걸 봄",
]

for item in test_inputs:
    poem = generate_poem(item)
    print("경험:", item)
    print(poem)
    print("line_count:", count_nonempty_lines(poem))
    print("---")


## 10. Adapter Hub 업로드

In [ ]:
from huggingface_hub import create_repo

create_repo(ADAPTER_REPO_ID, exist_ok=True)
model.push_to_hub(ADAPTER_REPO_ID)
tokenizer.push_to_hub(ADAPTER_REPO_ID)


## 11. Merged model 생성 및 업로드

In [ ]:
import gc
from huggingface_hub import create_repo
from transformers import AutoModelForCausalLM
from peft import PeftModel

try:
    del trainer
except NameError:
    pass
try:
    del model
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.bfloat16 if BF16_AVAILABLE else torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
merged_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
merged_model = merged_model.merge_and_unload()
merged_model.save_pretrained(MERGED_OUTPUT_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_OUTPUT_DIR)

create_repo(MERGED_REPO_ID, exist_ok=True)
merged_model.push_to_hub(MERGED_REPO_ID, safe_serialization=True)
tokenizer.push_to_hub(MERGED_REPO_ID)


## 12. 다음 단계

학습이 끝나면 Hugging Face Space backend의 `MODEL_ID`를 `z-unghyun/poem-generator-merged`로 설정하고 `/generate` 응답을 확인한다.
